# Preprocesamiento

Las señales crudas traen elementos que no forman parte de la actividad: el reposo antes de que la persona empiece a moverse, ruido de alta frecuencia y, en el acelerómetro, la gravedad mezclada con el movimiento. En este notebook se eliminan con tres operaciones: recorte del inicio, filtrado pasa-bajas y separación de la gravedad.

## Parámetros

In [ ]:
CARPETA = "datos/demo"
INDICE = 0   # grabación que se usa en los ejemplos

In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from scipy import signal

from wt9011 import cargar_grabaciones
from procesamiento import agregar_magnitudes, pasa_bajas, separar_gravedad, preprocesar

grabaciones = cargar_grabaciones(CARPETA)
for g in grabaciones:
    agregar_magnitudes(g["df"])

ETIQUETAS = sorted(set(g["etiqueta"] for g in grabaciones))
COLORES = {e: f"C{i}" for i, e in enumerate(ETIQUETAS)}

g = grabaciones[INDICE]
df = g["df"].copy()   # copia, para no modificar la grabación original en los ejemplos
fs = g["fs"]
print(f"Ejemplo: {g['archivo']}, fs = {fs:.2f} Hz")

## Recorte del inicio

Después de que arranca la grabación, la persona tarda alrededor de un segundo en empezar la actividad. Ese tramo de reposo no representa la actividad y se elimina.

In [ ]:
RECORTE = 1.0   # segundos

fig, ax = plt.subplots(figsize=(12, 3))
ax.plot(df["t"], df["a_mag"], lw=0.8, color=COLORES[g["etiqueta"]])
ax.axvspan(0, RECORTE, color="0.5", alpha=0.25, label="se elimina")
ax.set(xlabel="Tiempo [s]", ylabel="|a| [g]", title=g["archivo"])
ax.legend(loc="upper right")
ax.grid(alpha=0.3)
plt.show()

## Filtro pasa-bajas

El ruido del sensor y las vibraciones pequeñas ocupan frecuencias altas, mientras que el movimiento humano está casi todo por debajo de 10 Hz. Un filtro pasa-bajas deja pasar lo que está por debajo de la frecuencia de corte fc y atenúa lo que está arriba.

Se usa un filtro Butterworth, que tiene respuesta plana en la banda de paso, es decir, no deforma las frecuencias que deja pasar. En la frecuencia de corte la ganancia es 0.707 (−3 dB), lo que equivale a reducir la potencia a la mitad. El orden del filtro determina qué tan rápido atenúa después de fc. La gráfica muestra la respuesta en frecuencia para varios órdenes, hasta fs/2, que es la frecuencia más alta que se puede representar.

In [ ]:
FC = 10   # Hz

fig, ax = plt.subplots(figsize=(10, 3.5))
for orden in [1, 2, 4, 8]:
    sos = signal.butter(orden, FC, btype="low", fs=fs, output="sos")
    f, h = signal.sosfreqz(sos, worN=2048, fs=fs)
    ax.plot(f, np.abs(h), lw=1.5, label=f"orden {orden}")
ax.axvline(FC, color="k", lw=0.8, ls="--")
ax.axhline(1 / np.sqrt(2), color="k", lw=0.8, ls=":")
ax.text(FC + 0.3, 1.02, f"fc = {FC} Hz")
ax.text(0.3, 1 / np.sqrt(2) + 0.03, "0.707")
ax.set(xlabel="Frecuencia [Hz]", ylabel="Ganancia |H(f)|", xlim=(0, fs / 2), ylim=(0, 1.1))
ax.legend(loc="upper left", bbox_to_anchor=(1.01, 1))
ax.grid(alpha=0.3)
plt.show()

Con orden bajo la transición es gradual y todavía pasa una parte del ruido; con orden alto el corte es más abrupto. Orden 4 es un compromiso común.

El filtro aplicado a la velocidad angular en x, en un tramo de 2 segundos:

In [ ]:
ORDEN = 4
CANAL = "gx"
TRAMO = (3, 5)   # segundos

x = df[CANAL].to_numpy()
y = pasa_bajas(x, FC, fs, ORDEN)

m = (df["t"] >= TRAMO[0]) & (df["t"] <= TRAMO[1])
fig, ax = plt.subplots(figsize=(12, 3.5))
ax.plot(df["t"][m], x[m], lw=0.8, color="0.6", label="original")
ax.plot(df["t"][m], y[m], lw=1.5, color="C0", label=f"pasa-bajas {FC} Hz")
ax.set(xlabel="Tiempo [s]", ylabel=f"{CANAL} [°/s]")
ax.legend(loc="upper left", bbox_to_anchor=(1.01, 1))
ax.grid(alpha=0.3)
plt.show()

## Retraso del filtro

Un filtro que solo usa muestras pasadas, como el que se usaría en tiempo real, retrasa la señal. `sosfiltfilt` aplica el filtro hacia adelante y después hacia atrás: los dos retrasos se cancelan y la señal filtrada queda alineada con la original. Esto solo es posible cuando ya se tiene la grabación completa, que es el caso aquí. La función `pasa_bajas` de `procesamiento.py` usa esta versión.

In [ ]:
sos = signal.butter(ORDEN, FC, btype="low", fs=fs, output="sos")
causal = signal.sosfilt(sos, x)
fase_cero = signal.sosfiltfilt(sos, x)

fig, ax = plt.subplots(figsize=(12, 3.5))
ax.plot(df["t"][m], x[m], lw=0.8, color="0.6", label="original")
ax.plot(df["t"][m], causal[m], lw=1.5, color="C3", label="causal (sosfilt)")
ax.plot(df["t"][m], fase_cero[m], lw=1.5, color="C0", label="fase cero (sosfiltfilt)")
ax.set(xlabel="Tiempo [s]", ylabel=f"{CANAL} [°/s]")
ax.legend(loc="upper left", bbox_to_anchor=(1.01, 1))
ax.grid(alpha=0.3)
plt.show()

## Separación de la gravedad

El acelerómetro mide la suma de dos cosas: la gravedad, que solo cambia cuando cambia la orientación del sensor, y la aceleración del movimiento del cuerpo. Como la gravedad cambia despacio, se estima con un pasa-bajas de frecuencia de corte muy baja, 0.3 Hz. Al restarla de la señal original queda la aceleración del cuerpo, que oscila alrededor de cero.

In [ ]:
EJE = "az"

separar_gravedad(df, fs, fc=0.3)

fig, (ax1, ax2) = plt.subplots(2, 1, figsize=(12, 5), sharex=True)
ax1.plot(df["t"], df[EJE], lw=0.8, color="0.6", label="medida")
ax1.plot(df["t"], df[f"{EJE}_grav"], lw=1.5, color="C0", label="gravedad")
ax1.set(ylabel=f"{EJE} [g]", title=g["archivo"])
ax2.plot(df["t"], df[f"{EJE}_cuerpo"], lw=0.8, color="C1", label="cuerpo")
ax2.axhline(0, color="k", lw=0.8)
ax2.set(ylabel=f"{EJE} [g]", xlabel="Tiempo [s]")
for ax in (ax1, ax2):
    ax.legend(loc="upper left", bbox_to_anchor=(1.01, 1))
    ax.grid(alpha=0.3)
plt.tight_layout()
plt.show()

La separación es una aproximación. Si el sensor gira rápido, como en el balanceo del brazo al caminar, el cambio de orientación es más rápido que 0.3 Hz y una parte queda en la componente del cuerpo. Aun así, la aceleración del cuerpo ya no depende de cómo quedó colocado el sensor en reposo, y eso la hace más comparable entre grabaciones.

## Todo junto

La función `preprocesar` aplica las tres operaciones en orden a una grabación: recorta el inicio, filtra la aceleración y la velocidad angular con el pasa-bajas, separa la gravedad y calcula las magnitudes, incluida `cuerpo_mag`, la magnitud de la aceleración del cuerpo. Los ángulos no se filtran porque yaw salta de 180° a −180° y el filtro lo deformaría en ese salto.

In [ ]:
for g in grabaciones:
    g["pre"] = preprocesar(g["df"], g["fs"], recorte=RECORTE, fc=FC)

print("Columnas:", ", ".join(grabaciones[0]["pre"].columns))

# primera grabación de cada actividad
ejemplo = {}
for g in grabaciones:
    if g["etiqueta"] not in ejemplo:
        ejemplo[g["etiqueta"]] = g

VENTANA = 5   # segundos
fig, ejes = plt.subplots(len(ETIQUETAS), 1, figsize=(12, 2.2 * len(ETIQUETAS)), sharex=True, sharey=True)
for ax, etiqueta in zip(ejes, ETIQUETAS):
    d = ejemplo[etiqueta]["pre"]
    d = d[d["t"] <= VENTANA]
    ax.plot(d["t"], d["cuerpo_mag"], lw=0.8, color=COLORES[etiqueta])
    ax.set_ylabel(etiqueta)
    ax.grid(alpha=0.3)
ejes[0].set_title("Magnitud de la aceleración del cuerpo [g], ya preprocesada")
ejes[-1].set_xlabel("Tiempo [s]")
plt.tight_layout()
plt.show()

En `05_normalizacion` se ponen las señales en escalas comparables.